In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

from src.data_processing.data_loader import MovieLensDataLoader
from src.data_processing.splitters import DataSplitter
loader = MovieLensDataLoader()
data_dict = loader.load_data()
movie_id_to_idx = loader.get_movie_id_to_index()
splitter = DataSplitter(data_dict['ratings'])
split_data = splitter.leave_one_out()
train_data = split_data["train"].copy()
test_data = split_data["test"].copy()
train_data["movieIdx"] = train_data["movieId"].map(movie_id_to_idx)
test_data["movieIdx"] = test_data["movieId"].map(movie_id_to_idx)
train_data = train_data.dropna(subset = ["movieIdx"])
test_data = test_data.dropna(subset = ["movieIdx"])
train_data["movieIdx"] = train_data["movieIdx"].astype(int)
test_data["movieIdx"] = test_data["movieIdx"].astype(int)
def eval_loo(model, train_data, test_data, all_id, num_neg):
    hits = []
    ndcg = []
    user_h = train_data.groupby("userId")["timestamp"].apply(set).to_dict()
    for _, x in test_data.iterrows():
        user = x["userId"]
        movie = x["movieId"]
        hist = user_h.get(user, set())
        non_watched = list(all_id - hist - {movie})
        negative = np.random.choice(non_watched, num_neg, replace = False)
        eval = np.append(negative, movie)
        scores = model.get_scores(user, eval)
        rank_df = pd.DataFrame({'movie': eval, "score" : scores})
        rank_df = rank_df.sort_values(by = "score", ascending = False).reset_index()